# Criação da tabela Silver - CARDS

## Objetivos
- Criação de tabela que capture dados da bronze sem ocorrência de duplicatas e apenas de partidas encerradas
- Explode do array "cards" que contém as informações de cartões distribuídos nas partidas
- Seleção de dados do array
- Uso das informações do array para a criação de novas colunas

In [0]:
from pyspark.sql.functions import *

In [0]:
max_dh_bronze = (
    spark.read.table("api_football.bronze.matches_raw")
    # variável que captura a data mais recente de atualização (maior)
    .agg(max("dh_processing_br").alias("max_dh_bronze"))
    # transforma de dataframe para variável
    .collect()[0]["max_dh_bronze"]
)

# leitura da tabela de comparação, nesse caso a cards
df_silver = spark.read.table("api_football.silver.cards")

# join para pegar os dados que não estão na silver
df_matches_raw = (
    spark.read.table("api_football.bronze.matches_raw")
    .alias("mr")
    # left anti pega os dados das match_id que estão na bronze, mas não estão na silver
    .join(df_silver.alias("s"), col("s.match_id") == col("mr.match_id"), "leftanti")
    # filtra somente os dados que tem a data de atualização mais recente
    .filter(col("mr.dh_processing_br") == max_dh_bronze)
    # filtra somente as partidas finalizadas
    .filter(col("mr.match_status") == "Finished")
)

In [0]:
df_cards_exploded = df_matches_raw.select(
    "match_id",
    "match_referee",
    "match_hometeam_id",
    "match_hometeam_name",
    "match_awayteam_id",
    "match_awayteam_name",
    explode("cards").alias("card_event"),
    "dh_processing_br",
)
# display(df_cards_exploded)

In [0]:
df_cards = df_cards_exploded.select(
    col("match_id"),
    col("match_referee"),
    # verifica se o jogador que recebeu o cartão é do time da casa ou so visitante e retorna o nome do jogador na nova coluna player_name
    when(col("card_event.home_fault") != "", col("card_event.home_fault"))
    .otherwise(col("card_event.away_fault"))
    .alias("player_name"),
    # verifica se o jogador que recebeu o cartão é do time da casa ou so visitante e retorna o id do time na nova coluna team_id
    when(col("card_event.home_fault") != "", col("match_hometeam_id"))
    .otherwise(col("match_awayteam_id"))
    .alias("team_id"),
    # verifica se o jogador que recebeu o cartão é do time da casa ou so visitante e retorna o nome do time na nova coluna team_name
    when(col("card_event.home_fault") != "", col("match_hometeam_name"))
    .otherwise(col("match_awayteam_name"))
    .alias("team_name"),
    # verifica se o time que recebeu o cartão é o time da casa
    when(col("card_event.home_fault") != "", lit(True))
    .otherwise(lit(False))
    .alias("is_home_team"),
    # retorna o tipo do cartão, retorna em letras minusculas, criando a coluna card_type
    lower(col("card_event.card")).alias("card_type"),
    # retorna o tempo em que o cartão foi recebido, criando a coluna card_time
    col("card_event.time").alias("card_time"),
    # coluna que captura a data de atualização da bronze
    col("dh_processing_br").alias("dh_processing_bronze_br"),
    # coluna que captura a data de atualização do dataframe atual
    expr("current_timestamp()-interval 3 hours").alias("dh_processing_br"),
)

# display(df_cards)

In [0]:
df_cards.write.mode("append").saveAsTable("api_football.silver.cards")

In [0]:
%sql
-- select
--   *
-- from
--   api_football.silver.cards